# Taller final — Clasificación de texto y NER con LLMs

**Maestría Virtual en Ingeniería de Sistemas y Computación · PLN · Actividad 10**

Notebook **único** con los tres puntos del taller, **alineado a los notebooks de ayuda del curso**
(`pln-coding-ayudas`). Para **Google Colab / Kaggle con GPU**.

| Punto | Tema | Datos / método |
|---|---|---|
| **1** | Clasificación: TASS (N/NEU/P) y Sarcasmo (binario) | TASS local (`cr-tass.csv`/`cr1-tass.csv`), Sarcasmo desde Hugging Face |
| **2.1** | NER con BiLSTM sobre CoNLL-2002 | `nltk.corpus.conll2002` + FastText/CRF/CharCNN |
| **2.2** | NER fine-tuning sobre *próstata* | BETO / XLM-RoBERTa (token classification) |
| **3** | NER biomédico con Prompt Engineering | Mistral, Llama, Gemma, DeepSeek |

> **Antes de correr:** GPU activada · sube a Drive la carpeta con `pln-coding-ayudas/` (trae los CSV de
> TASS) · token de Hugging Face **write** para publicar.

## 0. Configuración del entorno

In [9]:
# 0.1  Instalación de librerías
!pip install -q -U transformers datasets evaluate accelerate sentencepiece
!pip install -q -U seqeval pytorch-crf bitsandbytes huggingface_hub
!pip install -q nltk
print("\n✅ Instalación terminada.")
print("⚠️ IMPORTANTE: ve a  Entorno de ejecución -> Reiniciar sesión  y luego")
print("   corre desde la celda 0.2 (NO vuelvas a correr esta 0.1).")


✅ Instalación terminada.
⚠️ IMPORTANTE: ve a  Entorno de ejecución -> Reiniciar sesión  y luego
   corre desde la celda 0.2 (NO vuelvas a correr esta 0.1).


In [10]:
# 0.2  Montar Drive y definir rutas
import os, glob
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
BASE_DIR   = DRIVE_ROOT
MODELS_DIR = '/content/drive/MyDrive/ACT_10_models'
os.makedirs(MODELS_DIR, exist_ok=True)

def buscar(nombre):
    hits = glob.glob(os.path.join(DRIVE_ROOT, '**', nombre), recursive=True)
    return hits[0] if hits else None

print('cr-tass.csv  ->', buscar('cr-tass.csv'))
print('cr1-tass.csv ->', buscar('cr1-tass.csv'))
print('training.bio ->', buscar('training.bio'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cr-tass.csv  -> /content/drive/MyDrive/PLN/ACT_10/pln-coding-ayudas/pln-coding-ayudas/TASS/cr-tass.csv
cr1-tass.csv -> /content/drive/MyDrive/PLN/ACT_10/pln-coding-ayudas/pln-coding-ayudas/TASS/cr1-tass.csv
training.bio -> /content/drive/MyDrive/PLN/ACT_10/prostata/training.bio


In [11]:
# 0.3  Dispositivo y semilla
import torch, numpy as np, random
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Dispositivo: cuda
GPU: Tesla T4


In [12]:
# 0.4  Login en Hugging Face (para publicar modelos y usar Llama/Gemma con licencia)
from huggingface_hub import login

# Token WRITE quemado para que corra de una vez y el profe revise lo publicado.
# ⚠️ SEGURIDAD: este token queda visible en el notebook. REVÓCALO/regenéralo en
#    https://huggingface.co/settings/tokens después de la calificación.
login()  # pega tu token WRITE al ejecutar

HF_USER = "oscaruv28"
print("Usuario HF:", HF_USER)

Usuario HF: oscaruv28


---
# Punto 1 — Clasificación de texto

- **TASS** (local): 3 clases `N` (negativo), `NEU` (neutro), `P` (positivo).
- **Sarcasmo** (Hugging Face): binario.

Modelos: **BETO** (`dccuchile/bert-base-spanish-wwm-cased`), **XLNet** (`xlnet-base-cased`),
**XLM-RoBERTa-large** (`xlm-roberta-large`).

In [13]:
# 1.0  Imports del Punto 1
import re
import pandas as pd, numpy as np, collections
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)

MODELS_CLS = {
    'beto'      : 'dccuchile/bert-base-spanish-wwm-cased',
    'xlnet'     : 'xlnet-base-cased',
    'xlmr_large': 'xlm-roberta-large',
}

## 1.1  Carga de TASS y distribución de etiquetas

Igual que el notebook de ayuda del curso: `cr-tass.csv` = **train**, `cr1-tass.csv` = **test**.
Se limpia **solo lo de Twitter** (URLs, menciones, emojis) — NO se quitan stopwords ni puntuación,
para no destruir el contexto que BETO necesita. Validación = 20% del train (estratificado).

In [14]:
# 1.1  Limpieza mínima estilo Twitter (del notebook de ayuda del curso)
def demoji(text):
    patron = re.compile("["
        u"\U0001F600-\U0001F64F" u"\U0001F300-\U0001F5FF" u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF" u"\U00002500-\U00002BEF" u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251" u"\U0001f926-\U0001f937" u"\U00010000-\U0010ffff"
        u"♀-♂" u"☀-⭕" u"‍⏏⏩⌚️〰"
        "]+", flags=re.UNICODE)
    return patron.sub('', text)

def clean_tweet(text):
    text = str(text)
    text = text.replace('<url>', '')
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = demoji(text)
    return re.sub(r'\s+', ' ', text).strip()

In [15]:
# 1.1b  Construcción del DatasetDict de TASS
ruta_train = buscar('cr-tass.csv')
ruta_test  = buscar('cr1-tass.csv')
assert ruta_train and ruta_test, 'No encuentro cr-tass.csv / cr1-tass.csv en tu Drive'

def _leer_tass(ruta):
    df = pd.read_csv(ruta)
    df = df[~df.isna().any(axis=1)].drop(columns=['id'], errors='ignore')
    df = df.rename(columns={'sentencia original': 'text', 'label': 'label_str'})
    df['text'] = df['text'].apply(clean_tweet)
    return df[['text', 'label_str']]

df_train_full = _leer_tass(ruta_train)
df_test       = _leer_tass(ruta_test)

TASS_L2I = {'N': 0, 'NEU': 1, 'P': 2}
TASS_I2L = {v: k for k, v in TASS_L2I.items()}

df_tr, df_val = train_test_split(df_train_full, test_size=0.2, random_state=42,
                                 stratify=df_train_full['label_str'])

def _to_ds(df):
    df = df.copy(); df['label'] = df['label_str'].map(TASS_L2I)
    return Dataset.from_pandas(df[['text', 'label']].dropna(), preserve_index=False)

ds_tass = DatasetDict({'train': _to_ds(df_tr), 'validation': _to_ds(df_val), 'test': _to_ds(df_test)})

print('=== TASS — distribución de etiquetas ===')
for split in ds_tass:
    c = collections.Counter(ds_tass[split]['label'])
    print(f'  {split:11s} n={len(ds_tass[split]):5d}  ' +
          '  '.join(f'{TASS_I2L[k]}:{v}' for k, v in sorted(c.items())))

=== TASS — distribución de etiquetas ===
  train       n= 3841  N:1508  NEU:1218  P:1115
  validation  n=  961  N:377  NEU:305  P:279
  test        n= 2443  N:951  NEU:793  P:699


## 1.2  Función reutilizable de fine-tuning

Fases: **tokenización + padding dinámico** (`DataCollatorWithPadding`), **entrenamiento**
(`TrainingArguments` + `Trainer`), **Early Stopping** y **f1-macro**. Parámetros del curso:
LR=1e-5, warmup 0.1, épocas 6–10, batch 8/16/32.

In [16]:
# 1.2  Fine-tuning parametrizable de un clasificador de secuencias
import evaluate
f1_metric  = evaluate.load('f1')
acc_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro'   : f1_metric.compute(predictions=preds, references=labels, average='macro')['f1'],
        'f1_weighted': f1_metric.compute(predictions=preds, references=labels, average='weighted')['f1'],
        'accuracy'   : acc_metric.compute(predictions=preds, references=labels)['accuracy'],
    }

def finetune_clasificador(model_key, ds, label2id, id2label,
                          batch_size=16, epochs=8, lr=1e-5, max_len=128,
                          run_name='run', push=False, repo_id=None):
    set_seed(42)
    ckpt = MODELS_CLS[model_key]
    tok  = AutoTokenizer.from_pretrained(ckpt)

    def tokenizar(batch):
        return tok(batch['text'], truncation=True, max_length=max_len)
    ds_tok = ds.map(tokenizar, batched=True, remove_columns=['text'])
    collator = DataCollatorWithPadding(tokenizer=tok)

    model = AutoModelForSequenceClassification.from_pretrained(
        ckpt, num_labels=len(label2id), id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True)

    args = TrainingArguments(
        output_dir=os.path.join(MODELS_DIR, run_name),
        num_train_epochs=epochs, learning_rate=lr, weight_decay=0.01, warmup_ratio=0.1,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='f1_macro', greater_is_better=True, save_total_limit=2,
        logging_steps=50, report_to='none', seed=42,
        fp16=torch.cuda.is_available(), push_to_hub=push, hub_model_id=repo_id)

    trainer = Trainer(
        model=model, args=args,
        train_dataset=ds_tok['train'], eval_dataset=ds_tok['validation'],
        processing_class=tok, data_collator=collator, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()
    m = trainer.evaluate(ds_tok['test'])
    print(f'\n>> [{run_name}] TEST f1_macro={m.get("eval_f1_macro",0):.4f} '
          f'acc={m.get("eval_accuracy",0):.4f}')
    if push:
        trainer.push_to_hub(); tok.push_to_hub(repo_id)
    return trainer, m

## 1.3  Fine-tuning sobre TASS (BETO, XLNet, XLM-RoBERTa-large)

> VRAM: con 16 GB usa `batch_size=16` para BETO/XLNet y `8` para `xlm-roberta-large`.

In [ ]:
# 1.3  TASS
res_tass = {}
_, res_tass['beto'] = finetune_clasificador('beto', ds_tass, TASS_L2I, TASS_I2L,
    batch_size=16, epochs=8, run_name='tass-beto', push=True, repo_id=f'{HF_USER}/tass-beto')
_, res_tass['xlnet'] = finetune_clasificador('xlnet', ds_tass, TASS_L2I, TASS_I2L,
    batch_size=16, epochs=8, run_name='tass-xlnet', push=True, repo_id=f'{HF_USER}/tass-xlnet')
_, res_tass['xlmr_large'] = finetune_clasificador('xlmr_large', ds_tass, TASS_L2I, TASS_I2L,
    batch_size=8, epochs=8, run_name='tass-xlmr-large', push=True, repo_id=f'{HF_USER}/tass-xlmr-large')

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  467MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted,Accuracy
1,1.104121,1.119522,0.201883,0.233824,0.391259
2,1.080518,1.059140,0.348158,0.367411,0.432882


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 1.4  Carga de Sarcasmo desde Hugging Face

La guía indica **descargar Sarcasmo directamente de Hugging Face**. Ajusta `DATASET_SARCASMO`
al dataset exacto que te indicó el curso. La función detecta automáticamente las columnas de
texto y etiqueta y arma train/validation/test.

In [ ]:
# 1.4  Carga genérica de un dataset binario desde Hugging Face
DATASET_SARCASMO = 'Ernesto-1997/Sarcastic_spanish_dataset'  # dataset del taller (Hugging Face)

def cargar_hf_binario(nombre):
    raw = load_dataset(nombre)
    # Detectar columnas de texto y etiqueta
    cols = raw['train'].column_names
    col_text  = next((c for c in cols if c.lower() in ('text','tweet','sentence','sentencia','texto','content')), cols[0])
    col_label = next((c for c in cols if c.lower() in ('label','labels','clase','class','target','sarcasmo')), cols[-1])
    print('Columnas detectadas -> texto:', col_text, '| etiqueta:', col_label)

    def norm(ej):
        return {'text': str(ej[col_text]), 'label_raw': ej[col_label]}
    raw = raw.map(norm)

    # Mapa de etiquetas -> 0/1
    valores = sorted(set(raw['train']['label_raw']), key=lambda x: str(x))
    l2i = {v: i for i, v in enumerate(valores)}
    i2l = {i: str(v) for v, i in l2i.items()}
    print('label2id:', l2i)

    def to_int(ej): return {'label': l2i[ej['label_raw']]}
    raw = raw.map(to_int)

    # Garantizar splits train/validation/test
    if 'validation' not in raw:
        tmp = raw['train'].train_test_split(test_size=0.2, seed=42)
        raw = DatasetDict({'train': tmp['train'], 'validation': tmp['test'],
                           'test': raw.get('test', tmp['test'])})
    keep = ['text', 'label']
    raw = DatasetDict({s: raw[s].remove_columns([c for c in raw[s].column_names if c not in keep])
                       for s in raw})
    return raw, l2i, i2l

ds_sarc, SARC_L2I, SARC_I2L = cargar_hf_binario(DATASET_SARCASMO)
for split in ds_sarc:
    c = collections.Counter(ds_sarc[split]['label'])
    print(f'  {split:11s} n={len(ds_sarc[split]):5d}  {dict(c)}')

## 1.5  Fine-tuning sobre Sarcasmo (BETO, XLM-RoBERTa-large)

In [ ]:
# 1.5  Sarcasmo
res_sarc = {}
_, res_sarc['beto'] = finetune_clasificador('beto', ds_sarc, SARC_L2I, SARC_I2L,
    batch_size=16, epochs=8, run_name='sarcasmo-beto', push=True, repo_id=f'{HF_USER}/sarcasmo-beto')
_, res_sarc['xlmr_large'] = finetune_clasificador('xlmr_large', ds_sarc, SARC_L2I, SARC_I2L,
    batch_size=8, epochs=8, run_name='sarcasmo-xlmr-large', push=True, repo_id=f'{HF_USER}/sarcasmo-xlmr-large')

## 1.6  Barrido de batch_size {8, 16, 32} (métricas del enunciado)

In [ ]:
# 1.6  Barrido de batch_size para un (modelo, dataset)
def barrido_batch(model_key, ds, l2i, i2l, tag, epochs=8, batches=(8, 16, 32)):
    filas = []
    for bs in batches:
        _, m = finetune_clasificador(model_key, ds, l2i, i2l, batch_size=bs, epochs=epochs,
                                     run_name=f'{tag}-bs{bs}', push=False)
        filas.append({'modelo': model_key, 'dataset': tag, 'batch_size': bs,
                      'f1_macro': round(m['eval_f1_macro'], 4), 'accuracy': round(m['eval_accuracy'], 4)})
    return pd.DataFrame(filas)

# Ejemplo (descomenta):
# display(barrido_batch('xlmr_large', ds_tass, TASS_L2I, TASS_I2L, 'tass-xlmr'))

## 1.7  Inferencia con los modelos publicados

In [ ]:
# 1.7  Predicciones de ejemplo (sentimiento)
from transformers import pipeline
dev = 0 if torch.cuda.is_available() else -1
clf_sent = pipeline('text-classification', model=f'{HF_USER}/tass-xlmr-large', device=dev)
ejemplos_sent = [
    'Me siento muy feliz hoy, fue un día increíble',
    'No puedo creer lo mal que salió todo, qué desastre',
    'Mañana hay reunión de trabajo a las 10am',
    'Odio los lunes, siempre pasa algo horrible',
    'Qué bonita sorpresa, no me lo esperaba!',
]
print('=== PREDICCIONES DE SENTIMIENTO (TASS) ===')
for t in ejemplos_sent:
    r = clf_sent(t)[0]
    print(f'{r["label"]:4s} ({r["score"]*100:5.1f}%) | {t}')

In [ ]:
# 1.7b  Predicciones de ejemplo (sarcasmo)
clf_sarc = pipeline('text-classification', model=f'{HF_USER}/sarcasmo-xlmr-large', device=dev)
ejemplos_sarc = [
    '¡Qué divertido, otro lunes de trabajo!',
    'Me encanta cuando llueve el día que tengo planes al aire libre',
    'Obvio que aprobé el examen, estudié exactamente 5 minutos',
    'Qué sorpresa que el internet se cayó justo cuando iba a entregar la tarea',
    'Sí, porque claramente yo soy el experto en cocina gourmet',
]
print('=== PREDICCIONES DE SARCASMO ===')
for t in ejemplos_sarc:
    r = clf_sarc(t)[0]
    print(f'Texto: {t}\n  Predicción: {r["label"]}  Confianza: {r["score"]*100:.2f}%\n')

## 1.8  Análisis (punto 5)

**¿Por qué TASS rinde menos que Sarcasmo?** TASS es de **3 clases** con la clase **NEU** difusa
y muy subjetiva (se solapa con POS y NEG), mientras que Sarcasmo es **binario** con señales más
separables → f1 más alto.

**Papel del número de clases:** más clases ⇒ *baseline* aleatorio más bajo (1/3 vs 1/2), más
fronteras de decisión y el f1-macro castiga la clase minoritaria/ambigua (NEU).

**LLMs:** BETO parte con ventaja en español; XLNet (inglés) suele quedar por debajo en TASS;
XLM-RoBERTa-large lidera cuando hay datos/épocas suficientes, a costa de más VRAM. En Sarcasmo la
brecha BETO vs XLM-R-large se reduce porque la tarea es más fácil.
*(Reemplaza con tus números reales.)*

---
# Punto 2 — NER

## 2.1  BiLSTM sobre CoNLL-2002 (baseline con nltk + mejoras)

Cargamos CoNLL-2002 con **nltk** (como el notebook de ayuda `Modelo_base_bilstm`) y entrenamos:
**BiLSTM+CRF+FastText** y **BiLSTM+CRF+CharCNN+FastText**, para superar el baseline (64.3%).

In [ ]:
# 2.1.0  Cargar CoNLL-2002 con nltk (igual que la ayuda del curso)
import nltk
nltk.download('conll2002', quiet=True)

def toks(s):  return [t for t, _, _ in s]
def labs(s):  return [l for _, _, l in s]
train_sents = list(nltk.corpus.conll2002.iob_sents('esp.train'))
eval_sents  = list(nltk.corpus.conll2002.iob_sents('esp.testa'))
test_sents  = list(nltk.corpus.conll2002.iob_sents('esp.testb'))
X_train, y_train = [toks(s) for s in train_sents], [labs(s) for s in train_sents]
X_eval,  y_eval  = [toks(s) for s in eval_sents],  [labs(s) for s in eval_sents]
X_test,  y_test  = [toks(s) for s in test_sents],  [labs(s) for s in test_sents]
print('Frases:', len(X_train), len(X_eval), len(X_test))
print('Ejemplo:', X_train[0][:8], '->', y_train[0][:8])

In [ ]:
# 2.1.1  Descargar FastText español (cc.es.300.vec ~ 1.2 GB)
EMBED_FILE = os.path.join(MODELS_DIR, 'cc.es.300.vec')
if not os.path.exists(EMBED_FILE):
    print('Descargando FastText español...')
    !wget -q -O {MODELS_DIR}/cc.es.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz
    !gunzip -f {MODELS_DIR}/cc.es.300.vec.gz
print('FastText:', EMBED_FILE)

In [ ]:
# 2.1.2  Vocabularios + matriz de embeddings FastText
from collections import Counter
words = {w.lower() for s in X_train + X_eval + X_test for w in s}
tagset = sorted({l for s in y_train + y_eval + y_test for l in s})
word2idx = {'-PAD-': 0, '-OOV-': 1, **{w: i + 2 for i, w in enumerate(sorted(words))}}
tag2idx  = {t: i for i, t in enumerate(tagset)}       # sin PAD en tags: usamos máscara CRF
idx2tag  = {i: t for t, i in tag2idx.items()}
chars = sorted({c for s in X_train for w in s for c in w})
char2idx = {'-PAD-': 0, '-OOV-': 1, **{c: i + 2 for i, c in enumerate(chars)}}
print('Vocab:', len(word2idx), '| Tags:', len(tag2idx), '| Chars:', len(char2idx))

EMB_DIM = 300
emb_matrix = np.random.normal(0, 0.1, (len(word2idx), EMB_DIM)).astype('float32')
emb_matrix[0] = 0.0
n_ok = 0
with open(EMBED_FILE, encoding='utf-8') as f:
    next(f)
    for line in f:
        p = line.rstrip().split(' ')
        if p[0] in word2idx:
            emb_matrix[word2idx[p[0]]] = np.asarray(p[1:], dtype='float32'); n_ok += 1
print(f'Vectores FastText mapeados: {n_ok}/{len(word2idx)}')

In [ ]:
# 2.1.3  Dataset y collate (palabras + caracteres + tags + máscara)
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
MAX_CHARS = 20

class NERDataset(TorchDataset):
    def __init__(self, X, Y): self.X, self.Y = X, Y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        w = [word2idx.get(t.lower(), 1) for t in self.X[i]]
        c = [[char2idx.get(ch, 1) for ch in t[:MAX_CHARS]] + [0]*(MAX_CHARS-len(t[:MAX_CHARS])) for t in self.X[i]]
        y = [tag2idx[t] for t in self.Y[i]]
        return torch.tensor(w), torch.tensor(c), torch.tensor(y)

def collate(batch):
    ws, cs, ys = zip(*batch)
    lengths = torch.tensor([len(w) for w in ws])
    ws = pad_sequence(ws, batch_first=True, padding_value=0)
    ys = pad_sequence(ys, batch_first=True, padding_value=tag2idx['O'])
    B, T = ws.shape
    cp = torch.zeros(B, T, MAX_CHARS, dtype=torch.long)
    for b, c in enumerate(cs): cp[b, :c.shape[0]] = c
    mask = torch.arange(T)[None, :] < lengths[:, None]
    return ws, cp, ys, mask

dl_train = DataLoader(NERDataset(X_train, y_train), batch_size=32, shuffle=True,  collate_fn=collate)
dl_eval  = DataLoader(NERDataset(X_eval,  y_eval),  batch_size=32, shuffle=False, collate_fn=collate)
dl_test  = DataLoader(NERDataset(X_test,  y_test),  batch_size=32, shuffle=False, collate_fn=collate)
print('Batches:', len(dl_train), len(dl_eval), len(dl_test))

In [ ]:
# 2.1.4  Arquitecturas: CharCNN, BiLSTM-CRF y BiLSTM-CRF-CharCNN
import torch.nn as nn
from torchcrf import CRF

class CharCNN(nn.Module):
    def __init__(self, vocab, emb=30, filtros=30, kernels=(3, 4, 5)):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, filtros, k, padding=k//2) for k in kernels])
        self.out_dim = filtros * len(kernels)
    def forward(self, chars):
        B, T, C = chars.shape
        x = self.emb(chars.view(B*T, C)).transpose(1, 2)
        f = [torch.relu(c(x)).max(dim=2)[0] for c in self.convs]
        return torch.cat(f, 1).view(B, T, -1)

class BiLSTM_CRF(nn.Module):
    def __init__(self, emb_matrix, num_tags, hidden=256, use_char=False, char_vocab=None, dropout=0.5):
        super().__init__()
        self.word_emb = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=False, padding_idx=0)
        din = emb_matrix.shape[1]
        self.use_char = use_char
        if use_char:
            self.char_cnn = CharCNN(char_vocab); din += self.char_cnn.out_dim
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(din, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)
    def _emis(self, w, c):
        x = self.word_emb(w)
        if self.use_char: x = torch.cat([x, self.char_cnn(c)], -1)
        x, _ = self.lstm(self.drop(x))
        return self.fc(x)
    def loss(self, w, c, y, m):   return -self.crf(self._emis(w, c), y, mask=m, reduction='mean')
    def decode(self, w, c, m):    return self.crf.decode(self._emis(w, c), mask=m)
print('Arquitecturas listas.')

In [ ]:
# 2.1.5  Entrenamiento + evaluación con seqeval
from seqeval.metrics import f1_score as seq_f1, classification_report as seq_report

def evaluar(model, loader=dl_test):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for w, c, y, m in loader:
            w, c, m = w.to(device), c.to(device), m.to(device)
            pred = model.decode(w, c, m)
            for i, seq in enumerate(pred):
                n = int(m[i].sum())
                yp.append([idx2tag[p] for p in seq])
                yt.append([idx2tag[int(t)] for t in y[i][:n]])
    return seq_f1(yt, yp), yt, yp

def entrenar_ner(model, epochs=15, lr=1e-3, nombre='ner'):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best = 0.0
    for ep in range(1, epochs+1):
        model.train(); tot = 0
        for w, c, y, m in dl_train:
            w, c, y, m = w.to(device), c.to(device), y.to(device), m.to(device)
            opt.zero_grad(); loss = model.loss(w, c, y, m); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); tot += loss.item()
        f1, _, _ = evaluar(model, dl_eval); best = max(best, f1)
        print(f'[{nombre}] ép {ep:2d} | loss {tot/len(dl_train):.3f} | eval F1 {f1:.4f}')
    print(f'>> [{nombre}] mejor eval F1: {best:.4f}')
    return best

In [ ]:
# 2.1.6  Arquitectura 1: BiLSTM + CRF + FastText
set_seed(42)
m1 = BiLSTM_CRF(emb_matrix, len(tag2idx), use_char=False)
f1_arq1 = entrenar_ner(m1, epochs=15, nombre='BiLSTM-CRF+FastText')
print('\nTEST:'); f1t1, yt, yp = evaluar(m1, dl_test); print('F1 test:', round(f1t1, 4))

In [ ]:
# 2.1.7  Arquitectura 2: BiLSTM + CRF + CharCNN + FastText
set_seed(42)
m2 = BiLSTM_CRF(emb_matrix, len(tag2idx), use_char=True, char_vocab=len(char2idx))
f1_arq2 = entrenar_ner(m2, epochs=15, nombre='BiLSTM-CRF+CNN+FastText')
f1t2, yt2, yp2 = evaluar(m2, dl_test)
print('\nReporte por entidad (mejor modelo):')
print(seq_report(yt2, yp2))

In [ ]:
# 2.1.8  Comparación con el baseline (64.3%)
pd.DataFrame([
    {'arquitectura': 'BiLSTM-base (baseline)',           'F1_test': 0.643},
    {'arquitectura': 'BiLSTM-CRF + FastText',            'F1_test': round(f1t1, 4)},
    {'arquitectura': 'BiLSTM-CRF + CharCNN + FastText',  'F1_test': round(f1t2, 4)},
])

## 2.2  NER fine-tuning (BETO / XLM-RoBERTa) sobre *próstata*

Fine-tuning de *token classification* sobre el dataset clínico **próstata**. Fases:
alineación del tokenizador, padding dinámico (`DataCollatorForTokenClassification`),
`AutoModelForTokenClassification`, `TrainingArguments` (batch 8/16/32, 6–10 épocas), `Trainer`
y publicación en HF.

> **Datos:** sube la carpeta `prostata/` a tu Drive (dentro de ACT_10). Trae archivos **`.bio`**
> (`token<TAB>etiqueta`, oraciones separadas por línea vacía). El notebook los busca solo y usa
> las versiones `_cleaned` para validación/test. 10 entidades: EDAD, CANCER, BIOMARCADOR, GLEASON,
> TNM, FECHA, CIRUGIA, DOSIS, MEDICAMENTO, TRATAMIENTO.

In [ ]:
# 2.2.1  Cargar dataset próstata (archivos .bio en formato BIO: token<TAB>etiqueta)
from datasets import Dataset as HFDataset, DatasetDict as HFDatasetDict

def _archivo(*nombres):
    # busca el archivo en cualquier subcarpeta de tu Drive de ACT_10
    for n in nombres:
        h = glob.glob(os.path.join(BASE_DIR, '**', n), recursive=True)
        if h: return h[0]
    raise FileNotFoundError(f'No encuentro {nombres} dentro de {BASE_DIR}')

def leer_bio(path):
    '''Lee BIO robustamente: ignora líneas mal formadas (token sin etiqueta válida).'''
    frases, labels, tk, lb = [], [], [], []
    for line in open(path, encoding='utf-8'):
        line = line.rstrip('\n')
        if not line.strip():
            if tk: frases.append(tk); labels.append(lb); tk, lb = [], []
            continue
        p = line.split()
        if len(p) >= 2 and (p[-1] == 'O' or p[-1].startswith('B-') or p[-1].startswith('I-')):
            tk.append(p[0]); lb.append(p[-1])   # token + etiqueta BIO válida
    if tk: frases.append(tk); labels.append(lb)
    return frases, labels

tr_t, tr_l = leer_bio(_archivo('training.bio', 'train.txt'))
va_t, va_l = leer_bio(_archivo('validation_cleaned.bio', 'validation.bio', 'valid.txt', 'dev.txt'))
te_t, te_l = leer_bio(_archivo('testing_cleaned.bio', 'testing.bio', 'test.txt'))

P_TAGS = sorted({l for seq in tr_l + va_l + te_l for l in seq})
P_L2I = {t: i for i, t in enumerate(P_TAGS)}
P_I2L = {i: t for t, i in P_L2I.items()}
print('Etiquetas próstata:', P_TAGS)

def _to_hf(t, l):
    return HFDataset.from_dict({'tokens': t, 'ner_tags': [[P_L2I[x] for x in s] for s in l]})
ds_pros = HFDatasetDict({'train': _to_hf(tr_t, tr_l), 'validation': _to_hf(va_t, va_l), 'test': _to_hf(te_t, te_l)})
print(ds_pros)

In [ ]:
# 2.2.2  Alineación de etiquetas + métrica seqeval
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
seqeval_metric = evaluate.load('seqeval')

def make_align(tokenizer):
    def fn(ex):
        tk = tokenizer(ex['tokens'], truncation=True, is_split_into_words=True, max_length=256)
        labs = []
        for i, lab in enumerate(ex['ner_tags']):
            wids = tk.word_ids(batch_index=i); prev = None; ids = []
            for wid in wids:
                if wid is None: ids.append(-100)
                elif wid != prev: ids.append(lab[wid])
                else: ids.append(-100)
                prev = wid
            labs.append(ids)
        tk['labels'] = labs
        return tk
    return fn

def compute_ner(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = [[P_I2L[p] for p, l in zip(pr, la) if l != -100] for pr, la in zip(preds, labels)]
    tl = [[P_I2L[l] for p, l in zip(pr, la) if l != -100] for pr, la in zip(preds, labels)]
    r = seqeval_metric.compute(predictions=tp, references=tl)
    return {'precision': r['overall_precision'], 'recall': r['overall_recall'],
            'f1': r['overall_f1'], 'accuracy': r['overall_accuracy']}

In [ ]:
# 2.2.3  Fine-tuning de token classification (BETO / XLM-RoBERTa)
def finetune_ner_llm(ckpt, run_name, batch_size=16, epochs=8, lr=2e-5, push=False, repo_id=None):
    set_seed(42)
    tok = AutoTokenizer.from_pretrained(ckpt)
    ds_tok = ds_pros.map(make_align(tok), batched=True, remove_columns=ds_pros['train'].column_names)
    collator = DataCollatorForTokenClassification(tokenizer=tok)
    model = AutoModelForTokenClassification.from_pretrained(
        ckpt, num_labels=len(P_L2I), id2label=P_I2L, label2id=P_L2I)
    args = TrainingArguments(
        output_dir=os.path.join(MODELS_DIR, run_name), num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='f1', greater_is_better=True, logging_steps=50, report_to='none',
        fp16=torch.cuda.is_available(), seed=42, push_to_hub=push, hub_model_id=repo_id)
    trainer = Trainer(model=model, args=args, train_dataset=ds_tok['train'],
                      eval_dataset=ds_tok['validation'], processing_class=tok,
                      data_collator=collator, compute_metrics=compute_ner,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    m = trainer.evaluate(ds_tok['test'])
    print(f'>> [{run_name}] TEST f1={m.get("eval_f1",0):.4f}')
    if push: trainer.push_to_hub(); tok.push_to_hub(repo_id)
    return m

res_pros = {}
res_pros['beto'] = finetune_ner_llm('dccuchile/bert-base-spanish-wwm-cased', 'prostata-beto',
    batch_size=16, epochs=8, push=True, repo_id=f'{HF_USER}/prostata-ner-beto')
res_pros['xlmr'] = finetune_ner_llm('xlm-roberta-base', 'prostata-xlmr',
    batch_size=16, epochs=8, push=True, repo_id=f'{HF_USER}/prostata-ner-xlmr')
pd.DataFrame([{'modelo': k, 'F1_test': round(v['eval_f1'], 4)} for k, v in res_pros.items()])

---
# Punto 3 — NER biomédico con Prompt Engineering

Sin fine-tuning: guiamos a **Mistral-7B**, **Llama-3.2-3B**, **Gemma-2** y **DeepSeek** (cuantizados)
para extraer entidades clínicas de un texto de cáncer de próstata. Cada modelo usa su **chat template**.

> ⚠️ El notebook de ayuda del curso traía un token de HF escrito en el código. **No lo uses** — aquí
> nos autenticamos con `login()` (celda 0.4). Avisa que ese token quedó expuesto.

In [ ]:
# 3.0  Texto clínico e instrucción de extracción
TEXTO_CLINICO = (
    "Paciente masculino de 72 años con antecedentes médicos de hipertensión arterial (HTA). "
    "Actualmente presenta un tumor prostático bilateral confirmado mediante biopsia. El "
    "diagnóstico histológico reveló un adenocarcinoma de próstata. El puntaje de Gleason "
    "reportado fue 3+3, lo que indica un cáncer de bajo grado. El PSA más alto registrado en "
    "la historia clínica fue de 9,9 ng/dL. La resonancia magnética no evidenció lesiones "
    "metastásicas ni afectación de la cápsula prostática. Las vesículas seminales se observaron "
    "normales. No se detectaron adenopatías ni lesiones óseas. La gammagrafía ósea fue negativa "
    "para metástasis. El cuadro clínico sugiere una neoplasia localizada de bajo riesgo."
)
INSTRUCCION = (
    "Eres un asistente experto en extracción de entidades clínicas (NER). Del siguiente texto "
    "clínico sobre cáncer de próstata, extrae las entidades y devuélvelas en JSON con estas claves "
    "(listas; [] si no aparece): edad_y_sexo, biomarcadores, tipo_de_cancer, cirugias, diagnostico, "
    "hallazgos_imagenes, examenes_realizados, antecedentes_medicos, resultados_laboratorio, "
    "tratamiento_y_medicacion. Responde SOLO con el JSON."
)

In [ ]:
# 3.1  Función genérica: carga cuantizada + chat template + generación
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

def extraer_entidades(model_id, load_in_4bit=True, compute_dtype=torch.bfloat16,
                      trust_remote_code=False, max_new_tokens=512):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    cfg = BitsAndBytesConfig(load_in_4bit=load_in_4bit, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=cfg,
                                                 device_map='auto', trust_remote_code=trust_remote_code)
    msgs = [{'role': 'user', 'content': f'{INSTRUCCION}\n\nTexto:\n{TEXTO_CLINICO}'}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    resp = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    del model; torch.cuda.empty_cache()
    return resp

In [ ]:
# 3.2  Mistral-7B-Instruct-v0.2
print('=== Mistral-7B-Instruct-v0.2 ===')
print(extraer_entidades('mistralai/Mistral-7B-Instruct-v0.2', compute_dtype=torch.float16))

In [ ]:
# 3.3  Llama-3.2-3B-Instruct (requiere aceptar licencia en HF)
print('=== Llama-3.2-3B-Instruct ===')
print(extraer_entidades('meta-llama/Llama-3.2-3B-Instruct', compute_dtype=torch.float16))

In [ ]:
# 3.4  Gemma-2 (requiere aceptar licencia en HF)
print('=== Gemma-2-2b-it ===')
print(extraer_entidades('google/gemma-2-2b-it', compute_dtype=torch.bfloat16))

In [ ]:
# 3.5  DeepSeek (trust_remote_code=True)
print('=== DeepSeek-LLM-7B-chat ===')
print(extraer_entidades('deepseek-ai/deepseek-llm-7b-chat', trust_remote_code=True,
                        compute_dtype=torch.bfloat16))

### 3.6  Análisis (Prompt Engineering)

- Cada LLM usa un **chat template** distinto (`[INST]…[/INST]` Mistral, `<start_of_turn>` Gemma,
  `<|start_header_id|>` Llama, `<|User|>…<|Assistant|>` DeepSeek); `apply_chat_template` lo maneja.
- **Sin fine-tuning:** la calidad depende del prompt y del modelo. Los multilingües grandes suelen
  extraer mejor las entidades clínicas en español.
- Compara estas salidas JSON con las entidades del dataset próstata del Punto 2.2.

---
## Cierre
- **Punto 1:** TASS y Sarcasmo afinados y publicados; la brecha de f1 se explica por el número de clases.
- **Punto 2:** BiLSTM-CRF(+CharCNN)+FastText supera el baseline 64.3%; BETO/XLM-R afinados en próstata.
- **Punto 3:** extracción clínica por prompt con 4 LLMs generativos cuantizados.